# 05 · 계산기와 검색을 붙인 단일 루프
**4세션 · 도구 결과도 컨텍스트다** — 실습 40분

모델은 *이 도구를 이 값으로 불러 달라*고 말할 뿐이고, **실행은 우리 코드가 합니다.**
시도마다 도구 결과(`tool_result`)가 messages에 쌓입니다 — 그것이 다음 시도에서 모델이 읽는 컨텍스트입니다.

에이전트 = **모델 + 도구 + 루프**

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")          # day 폴더로 이동 — data/ · prompts/ · src/ 가 여기 있다

from common import llm      # 공용 API 도구 (ETRI_LLM_3days/common/llm.py)
if llm.MOCK:
    print("모델:", llm.MODEL, "| 모의 모드")
else:
    print("모델:", llm.MODEL, "| 실제 호출")

from src import rag
if rag.EMBED == "hash":
    print("임베딩: 간이 해시 (EMBED=hash)")
else:
    print("임베딩:", rag.ST_MODEL)

import json
import inspect
from src import loop        # 검색 색인(loop.SEARCH_INDEX)도 loop.py 안에서 만든다

## 1. 함수 정의의 실물
이름 · 설명 · 입력 스키마 세 가지가 전부입니다. **설명은 모델이 도구를 고르는 유일한 근거**입니다.

In [ ]:
print(json.dumps(loop.TOOLS, ensure_ascii=False, indent=2))

> 문자열 수식을 받아 `eval`로 계산하는 도구는 만들지 않습니다. 모델이 무엇을 넘길지 우리가 통제할 수 없기 때문입니다. `enum`으로 고를 수 있는 값을 제한합니다.

## 2. 루프 코드

In [ ]:
print(inspect.getsource(loop.run))

## 3. 1단계 · 붙이기
질문 네 개로 돌려 **시도마다 출력**을 봅니다. 줄 앞의 `[0]` `[1]` `[2]`는 몇 번째 모델 호출인지입니다.
- 셋째 질문 — 두 도구를 차례로 부르는가?
- 넷째 질문 — 검색 → 곱셈 → 덧셈처럼, **앞 도구의 결과를 봐야 다음 도구를 부를 수 있는** 일을 몇 번의 시도로 푸는가?

In [ ]:
Q1 = "AX-2041의 무게는 얼마인가요?"
Q2 = "1,234,500원을 3명이 똑같이 나누면 1명당 얼마인가요?"
Q3 = "서울로 3박 출장을 가면 숙박비 한도는 모두 얼마인가요?"
Q4 = "AX-2041 10대와 AX-2014 5대를 한 상자에 담으면 제품 무게만 모두 몇 kg인가요?"

for q in [Q1, Q2, Q3, Q4]:
    print("=" * 70)
    print("Q", q)
    answer, trace = loop.run(q)
    print("→ 최종 답:", answer)
    print("→ 쓴 도구:", loop.tools_used(trace))
    print()

## 4. 2단계 · 일부러 틀리게
시스템 프롬프트와 코드는 그대로 두고, **search 도구의 입력(query) 설명 한 줄만** 잘못 적습니다.
문서는 한국어인데 *"영어 단어 하나로 찾아라"*고 적었습니다. 모델은 도구 정의를 믿고 그대로 따릅니다 — 정의가 틀리면 충실하게 틀립니다.

볼 것: 모델이 어떤 검색어를 넣는가 · 못 찾으면 몇 번 다시 시도하는가 · 최종 답이 나오는가

In [ ]:
print("좋은 정의의 query 설명:", loop.SEARCH_TOOL["input_schema"]["properties"]["query"]["description"])
print("잘못된 정의의 query 설명:", loop.SEARCH_TOOL_BAD["input_schema"]["properties"]["query"]["description"])

TRY = [
    "자기계발비 한도를 12개월로 나누면 한 달에 얼마인가요?",
    Q3,
    "회의실은 하루에 몇 번 예약할 수 있나요?",
]
for q in TRY:
    print("=" * 70)
    print("Q", q)
    answer, trace = loop.run(q, tools=loop.TOOLS_BAD)
    print("→ 최종 답:", answer)
    print("→ 쓴 도구:", loop.tools_used(trace))
    print()

## 5. 정의 한 줄만 고쳐 바로잡는다
4절에서 실패한 질문을 하나 정하고, **query의 description만** 고쳐 다시 돌립니다. 코드 · 시스템 프롬프트는 건드리지 않습니다.

In [ ]:
MY_SEARCH_TOOL = {
    "name": "search",
    "description": loop.SEARCH_TOOL["description"],
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string",
                      "description": "English keyword only, one word (e.g. 'hotel')"},   # ✏️ 이 문장만 고친다
        },
        "required": ["query"],
    },
}
MY_TOOLS = [loop.CALCULATOR_TOOL, MY_SEARCH_TOOL]
Q = TRY[0]                                                    # ✏️ 4절에서 실패한 질문

print("=" * 70)
print("[고치기 전]")
answer, trace = loop.run(Q, tools=loop.TOOLS_BAD)
print("→ 최종 답:", answer)
print("→ 쓴 도구:", loop.tools_used(trace))

print("=" * 70)
print("[고친 뒤]")
answer, trace = loop.run(Q, tools=MY_TOOLS)
print("→ 최종 답:", answer)
print("→ 쓴 도구:", loop.tools_used(trace))

| 질문 | 고치기 전 검색어 · 결과 | 고친 query 설명 | 고친 뒤 검색어 · 결과 |
|---|---|---|---|
| | | | |

> 참고 — 같은 실험에서 *언제 쓰는지*(description)를 "검색한다." 한 마디로 줄이거나 엉뚱하게 바꿔도, 시스템 프롬프트가 있으면 모델은 대부분 제대로 골랐습니다. 이름과 문맥이 보완해 주기 때문입니다. **입력값을 어떻게 채우라는 정의**는 모델이 그대로 따르므로, 틀리면 바로 결과가 무너집니다.

## 6. 창에 무엇이 쌓였나
`loop.run()`이 모델에 보낸 대화 기록(messages)을 펼쳐 봅니다.
시도마다 **모델의 도구 요청**과 **우리가 붙인 도구 결과**가 쌓이고, 마지막 모델 호출에는 이 전부가 들어갑니다 — 그래서 **도구 결과도 컨텍스트**입니다.

In [ ]:
answer, trace = loop.run(Q3, verbose=False)

loop.show_messages(loop.LAST_MESSAGES)      # [번호] = messages 안의 순서
print()
print("최종 답:", answer)

## 내 loop.py에 없는 것 → 내일

| 전략 | 오늘의 loop.py | Claude Code |
|---|---|---|
| 고르기 | 내가 짠 검색 하나 | 파일 검색 도구 · 스킬은 쓸 때만 본문을 불러온다 |
| 줄이기 | 없음 — messages가 끝없이 쌓인다 | 오래된 도구 출력부터 비우고, 필요하면 대화를 요약한다 |
| 쓰기 | 없음 — 끝나면 사라진다 | CLAUDE.md · 자동 메모리를 세션마다 읽어 온다 |
| 나누기 | 없음 — 창이 하나뿐 | 서브에이전트가 자기 창에서 일하고 요약만 돌려준다 |

내일은 오른쪽 열을 다 갖춘 루프 위에서 시작합니다.